# CS4100 Final Project  
Team Members: Khushi Khan, Dustin Zhang, Kayla Handley, Koena Gupta

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import random

from torch import nn
from torch import optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torchmetrics.classification import MultilabelAccuracy, MultilabelPrecision, MultilabelRecall, MultilabelF1Score
from sklearn.metrics import confusion_matrix

In [ ]:
mp3_df = pd.read_csv('../data/cleaned/fma_cleaned_dataset_emotion_labels.csv', low_memory=False)

mp3_df.head()

In [ ]:
track_ids_df = np.load('../data/cleaned/fma_track_ids.npy')
spectrograms_df = np.load('../data/cleaned/fma_spectrograms.npy')
labels_df = np.load('../data/cleaned/fma_labels.npy') # dont know if we need this

# Inspecting the data
for i in range(3):
    track_id = track_ids_df[i]
    print(f"Track id: {track_id}")
    print(f"Labels: {labels_df[i]}")
    print(f"Valence: {mp3_df.loc[mp3_df['track_id'] == track_id, 'valence'].item()}")
    print(f"Energy: {mp3_df.loc[mp3_df['track_id'] == track_id, 'energy'].item()}")
    plt.imshow(spectrograms_df[i], cmap='gray')
    plt.show()

In [ ]:
# Shuffling data
random.seed(42)

track_ids_li = [int(track_id) for track_id in track_ids_df]

indices = np.random.permutation(len(track_ids_li))

shuffled_track_ids = track_ids_df[indices]
shuffled_spectrograms = spectrograms_df[indices]

# Targets to predict
y = mp3_df[['emotion_joy_excitement_softmax', 
            'emotion_peaceful_content_softmax', 
            'emotion_anger_tension_softmax', 
            'emotion_sadness_softmax']].to_numpy()

# Converting targets to binary for multi label classification
y = (y > 0.25).astype(np.float32)

n_outputs = y.shape[1]

print(f"Expected number of outputs: {n_outputs}")

# Passing in spectrograms
# X = spectrograms_df
X = shuffled_spectrograms
n_samples, n_mels, n_timeframes = X.shape
X = X.reshape(n_samples, 1, n_mels, n_timeframes)
n_samples, num_channels, n_mels, n_timeframes = X.shape

shape_info = {
    "Number of samples": n_samples,
    "Number of channels": num_channels,
    "Number of Mel frequency bands": n_mels,
    "Number of time frames": n_timeframes
}

for k, v in shape_info.items():
    print(f"{k}: {v}")

In [ ]:
emotion_counts = {
    'Joy-Excitement': y[:, 0].sum(), 
    'Peaceful-Content': y[:, 1].sum(), 
    'Anger-Tension': y[:, 2].sum(), 
    'Sadness': y[:, 3].sum()
}

plt.bar(emotion_counts.keys(), emotion_counts.values())
plt.xlabel("Emotions")
plt.ylabel("Emotion Count")
plt.title("Emotional Distribution Over All Songs")
plt.show()

According to the distribution, the emotional classes in our target data are balanced. 

In [ ]:
# Defining a custom dataset class
class SpectrogramDataset(Dataset):
    def __init__(self, spectrograms, labels, transform=None, resize=None):
        self.spectrograms = spectrograms
        self.labels = labels
        self.transform = transform
        self.resize = resize

    def __len__(self):
        return len(self.spectrograms)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img = self.spectrograms[idx]
        emotion_labels = self.labels[idx]
        
        # Convert numpy array/image to Pytorch tensor
        if self.transform:
            img = self.transform(img)

        # Convert targets to Pytorch tensor type
        emotion_labels = torch.tensor(emotion_labels, dtype=torch.float32)

        if not torch.is_tensor(img) or not torch.is_tensor(emotion_labels):
            raise TypeError("Expected a torch.Tensor: Either the spectrogram or labels are incorrect types")
        
        if img.shape[1] != 1:
            n_timeframes, n_channels, n_mels = img.shape
            img = img.reshape(n_channels, n_timeframes, n_mels)

        # Returning (spectrogram, emotional labels)
        return img, emotion_labels

In [ ]:
# Splitting data into train, val, and test sets manually to preserve insertion order
X_split_idx = int(X.shape[0] * 0.6)
y_split_idx = int(y.shape[0] * 0.6)
X_train, X_other = X[:X_split_idx, :], X[X_split_idx:, :] # 60% training, 40% other (for val and test)
y_train, y_other = y[:y_split_idx, :], y[y_split_idx:, :] # 60% training, 40% other (for val and test)

X_val_split_idx = int(X_other.shape[0] * 0.5)
y_val_split_idx = int(y_other.shape[0] * 0.5)

X_val, X_test = X[:X_val_split_idx, :], X[X_val_split_idx:, :] # 50% validation, 50% test
y_val, y_test = y[:y_val_split_idx, :], y[y_val_split_idx:, :] # 50% validation, 50% test

In [ ]:
# Initialize the datasets
img_dims = (256, 256)
transform = transforms.ToTensor()
train_ds = SpectrogramDataset(spectrograms=X_train, 
                              labels=y_train, 
                              transform=transform, 
                              resize=img_dims)
val_ds = SpectrogramDataset(spectrograms=X_val, 
                             labels=y_val,
                             transform=transform, 
                             resize=img_dims)
test_ds = SpectrogramDataset(spectrograms=X_test, 
                             labels=y_test,
                             transform=transform, 
                             resize=img_dims)

# Initialize the data loaders; insertion order matters, so we won't shuffle
batch_size = 10
train_dataloader = DataLoader(dataset=train_ds, 
                              batch_size=batch_size, 
                              shuffle=False)
test_dataloader = DataLoader(dataset=test_ds, 
                             batch_size=batch_size, 
                             shuffle=False)
val_dataloader = DataLoader(dataset=val_ds,
                            batch_size=batch_size,
                            shuffle=False)

In [ ]:
img, labels = train_ds[0]
print(img.shape)

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_channels=1, n_outputs=4):
        super(CNN, self).__init__()

        # 1st convolutional layer
        self.conv1 = nn.Conv2d(
            in_channels=in_channels, 
            out_channels=16,
            kernel_size=3,
            padding=1)
        
        # Max pooling layer 
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 2nd convolutional layer
        self.conv2 = nn.Conv2d(
            in_channels=16, 
            out_channels=32, 
            kernel_size=3,
            padding=1)
        
        # 3 convolutional layer
        self.conv3 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        # Fully connected layer
        self.n_outputs = n_outputs
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(64, n_outputs)

    def forward(self, x):
        # print(f"Forward pass x shape: {x.shape}")
        n_batch, n_time_frames, n_channels, n_mels = x.shape
        x = x.reshape(n_batch, n_channels, n_mels, n_time_frames)

        x = F.relu(self.conv1(x))  # Apply first convolutional layer and ReLU activation func
        x = self.pool(x)           # Apply max pooling

        x = F.relu(self.conv2(x))  # Apply second convolutional layer and ReLU activation func
        x = self.pool(x)           # Apply max pooling

        x = F.relu(self.conv3(x))  # Apply third convolutional layer and ReLU activation func
        x = self.pool(x)
        
        x = self.global_pool(x)
        x = x.reshape(x.shape[0], -1)  # Flatten the tensor

        x = self.fc1(x)
        return x

device = "cuda" if torch.cuda.is_available() else "cpu"

# Since the spectrograms are grayscale, in-channels=1
model = CNN(in_channels=1, n_outputs=len(emotion_counts.keys())).to(device)
    
print(f"Model architecture:\n {model}")

In [ ]:
# Define the loss function for multi-label classification
criterion = nn.BCEWithLogitsLoss()

# Define the optimizer and learning rate
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs=20
train_losses = []
val_losses = []
for epoch in range(num_epochs):
  # Iterate over training batches
  print(f"Epoch [{epoch + 1}/{num_epochs}]")

  sum_loss = 0
  loss_len = 0
  for _, batch in enumerate(tqdm(train_dataloader)):
    # Move data and targets to GPU, faster performance
    data, targets = batch # Expected data shape: (samples, channels, mels, time frames)

    data = data.to(device)
    targets = targets.to(device)

    # Accumulate gradients
    optimizer.zero_grad()

    # Predicted output
    preds = model(data)

    # Calculating Binary Cross Entropy loss
    loss = criterion(preds, targets)

    # Computes the gradients of the loss w.r.t. model parameters/Backward pass
    loss.backward()

    # Update the weights
    optimizer.step()

    # Track loss in one epoch
    sum_loss += loss.item()
    loss_len += 1

  sum_valid_loss = 0
  valid_loss_len = 0
  for _, batch in enumerate(val_dataloader):
     # Move data and targets to GPU, faster performance
    data, targets = batch # Expected data shape: (samples, channels, mels, time frames)

    data = data.to(device)
    targets = targets.to(device)

    # Predicted output
    preds = model(data)

    # Calculating Binary Cross Entropy loss
    loss = criterion(preds, targets)

    # Track loss in one epoch
    sum_valid_loss += loss.item()
    valid_loss_len += 1

  train_losses.append(sum_loss / loss_len)
  val_losses.append(sum_valid_loss / valid_loss_len)

In [ ]:
plt.plot(np.arange(0, num_epochs, 1), train_losses, label="Training Loss")
plt.plot(np.arange(0, num_epochs, 1), val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Average Loss per Epoch")
plt.title(f"CNN Loss over {num_epochs} Epochs")
plt.legend()
plt.grid(True)
plt.show()

Based on the plot, the loss falls drastically by epoch 4, then levels out by epoch 13. The loss isn't greatly impacted by epoch 15. This means the model has reached its capacity to extract pattern from the data by epoch 13. Continuing to train the model beyoond this point results in diminishing returns. 

In [ ]:
# Set up of multilabel accuracy metric
accuracy = MultilabelAccuracy(num_labels=n_outputs, average="macro")
precision = MultilabelPrecision(num_labels=n_outputs, average="macro")
recall = MultilabelRecall(num_labels=n_outputs, average="macro")
f1 = MultilabelF1Score(num_labels=n_outputs, average="macro")
f1_by_emotion = MultilabelF1Score(num_labels=n_outputs, average="none")
accuracy_by_emotion = MultilabelAccuracy(num_labels=n_outputs, average="none")

scores = { "Accuracy": [], "Precision": [], "Recall": [], "F1": [] }
emotions = ["Joy-Excitement", "Peaceful-Content", "Anger-Tension", "Sadness"]
all_preds = []
all_targets = []

# Iterate over the dataset batches
model.eval()
with torch.no_grad():
        for _, batch in enumerate(tqdm(test_dataloader)):
                data, targets = batch # Expected data shape: (samples, channels, mels, time frames)

                # Get predictions using test data, convert them to appropriate form
                logits = model(data)
                probs = torch.sigmoid(logits)

                preds = (probs > 0.3).int()
                targets = targets.int()

                all_preds.append(preds.cpu())
                all_targets.append(targets.cpu())

                accuracy(preds, targets)
                precision(preds, targets)
                recall(preds, targets)
                f1(preds, targets)

        # Record test dataset performance
        scores = { 
                "Accuracy": accuracy.compute().item(), 
                "Precision": precision.compute().item(), 
                "Recall": recall.compute().item(), 
                "F1": f1.compute().item()
        }


f1_values = f1_by_emotion.compute().cpu().numpy()
f1_per_emotion = {emotions[i]: f1_values[i] for i in range(n_outputs)}

all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

accuracy_per_emotion = (all_preds == all_targets).float().mean(dim=0).numpy()
accuracy_per_emotion = {
    emotions[i]: accuracy_per_emotion[i] for i in range(n_outputs)
}

In [ ]:
# Plotting scores for each metric
for metric, score in scores.items():
    print(f"{metric}: {score}")
    plt.bar(metric, score)

plt.legend(scores.keys())
plt.xlabel("Metrics")
plt.ylabel("Scores")
plt.title("CNN Accuracy-Precision-Recall-F1 Scores on Test Data")
plt.show()

In [ ]:
# Plotting accuracy scores for each emotion
for emotion, score in accuracy_per_emotion.items():
    print(f"{emotion}: {score}")
    plt.bar(emotion, score)

plt.legend(accuracy_per_emotion.keys())
plt.xlabel("Emotions")
plt.ylabel("Accuracy Scores")
plt.title("CNN Accuracy Per Emotion")
plt.show()

In [ ]:
# Plotting f1 scores for each emotion
for emotion, score in f1_per_emotion.items():
    print(f"{emotion}: {score}")
    plt.bar(emotion, score)

plt.legend(f1_per_emotion.keys())
plt.xlabel("Emotions")
plt.ylabel("F1 Scores")
plt.title("CNN F1 Score Per Emotion")
plt.show()